# Model Quantization

> In Section 19 we worked through an INT8 symmetric quantization by hand: pick a scale, divide the floating-point weights by it, round, then multiply the scale back to recover them. But when the weight distribution is asymmetric (for example, an attention output that is almost entirely non-negative), symmetric quantization wastes half of the integer range on the negative side that is never used. This section expands on the quantization algorithm itself: symmetric vs. asymmetric, per-tensor vs. per-channel vs. per-group, and the problems that calibration-based methods such as GPTQ and AWQ are designed to solve.
>
> The goal is to make one thing clear: modern LLMs can run at 4-bit not by "directly rounding", but by first handling the weight distribution and then quantizing.


The core question quantization has to answer is this: when we approximate a range of floating-point numbers with a smaller set of integers, is the error acceptable? This error comes from two places — magnitude (which range the numbers fall in) and shape (how the numbers are distributed within that range).

Symmetric quantization assumes the range is symmetric around 0. The formula is simple, but it loses out on shifted distributions. Asymmetric quantization introduces a zero point to translate the range so it fits the real distribution better, at the cost of one extra parameter. This is the first layer of choice.

The second layer of choice is granularity. Sharing one scale across an entire layer is simple to implement, but different rows (different output channels) can have very different numerical ranges, and accommodating them all with one scale amplifies the error. Giving each channel its own scale solves this, but adds parameters. Going one step finer, splitting each row into small groups with one scale per group gives the group-wise quantization that modern 4-bit quantization typically uses.

These two layers of choice together explain why RTN (Round-To-Nearest, direct rounding) works poorly on LLMs — the activation distribution in LLMs contains a few outlier channels (channels with extremely large values). Their range stretches a single scale, compressing all the normal channels into a handful of integer levels. GPTQ and AWQ are two algorithms that tackle this outlier problem from different angles.


## 1. Review and Limitations of Symmetric Quantization

We first use a tiny code snippet to review the symmetric quantization from Section 19 and observe its limitation: when all the data is non-negative, the negative side of symmetric quantization is completely wasted.


In [ ]:
import numpy as np

np.random.seed(42)

# Simulate a piece of post-ReLU activation: all non-negative, mostly small, with a few extreme values
x = np.abs(np.random.randn(8).astype(np.float32)) * 0.3
x[0] = 5.0  # one outlier
print('Original data (all non-negative):')
print(np.round(x, 3))

# Symmetric quantization: scale = max(|x|) / 127
scale_sym = np.max(np.abs(x)) / 127
x_q_sym = np.round(x / scale_sym).clip(-127, 127).astype(np.int8)
x_deq_sym = x_q_sym.astype(np.float32) * scale_sym

print(f'\nSymmetric quantization scale = {scale_sym:.4f}')
print(f'Quantized integers: {x_q_sym}')
print(f'Dequantized recovery: {np.round(x_deq_sym, 3)}')
print(f'\nKey observation: the negative integer side (-127..-1) is never used — half of the integer range is wasted.')


## 2. Asymmetric Quantization: Introducing the Zero Point

Asymmetric quantization changes the mapping range from $[-127, 127]$ to $[0, 255]$, and introduces a zero point to align with the data range. The formula becomes:

$$q = \text{round}\left(\frac{x - x_{\min}}{s}\right), \quad s = \frac{x_{\max} - x_{\min}}{255}$$

where $x_{\min}$ and $x_{\max}$ are the actual minimum and maximum of this data segment. During recovery: $\hat{x} = q \cdot s + x_{\min}$.

Let us work through the non-negative data from the previous section by hand.


In [ ]:
# Asymmetric quantization
x_min, x_max = float(x.min()), float(x.max())
scale_asym = (x_max - x_min) / 255
zero_point = x_min  # actual minimum of the data

x_q_asym = np.round((x - zero_point) / scale_asym).clip(0, 255).astype(np.uint8)
x_deq_asym = x_q_asym.astype(np.float32) * scale_asym + zero_point

print(f'Asymmetric quantization scale = {scale_asym:.4f}, zero_point = {zero_point:.4f}')
print(f'Quantized integers (uint8): {x_q_asym}')
print(f'Dequantized recovery: {np.round(x_deq_asym, 3)}')

mae_sym = np.mean(np.abs(x - x_deq_sym))
mae_asym = np.mean(np.abs(x - x_deq_asym))
print(f'\nMAE comparison:')
print(f'  Symmetric:   {mae_sym:.4f}')
print(f'  Asymmetric: {mae_asym:.4f}')
print(f'\nKey observation: the integer range [0, 255] is fully utilized, and the MAE is typically lower.')


### Tradeoffs Between the Two Schemes

| Scheme | Formula | Parameters | Suitable scenario |
|:---|:---|:---|:---|
| Symmetric | $q = \text{round}(x / s)$, $s = \max|\cdot|/127$ | one scale | weights (usually symmetric around 0) |
| Asymmetric | $q = \text{round}((x - z) / s)$ | scale + zero_point | activations (often shifted) |

Engineering cost difference: asymmetric quantization adds one extra addition during matrix multiplication (to handle the zero_point), but usually gets back a noticeably lower error. INT8/INT4 kernels in modern inference frameworks (such as llama.cpp and vLLM) support both, and choose based on the tensor property.


## 3. Quantization Granularity: per-tensor / per-channel / per-group

Above we used the setting of "one scale shared by the entire data segment". In practice, each row (each output channel) of a weight matrix can have very different numerical ranges — some rows have generally small weights, others have large values. If they share one scale, the rows with large values push the scale up, and the precision of the small-value rows is sacrificed.

This leads to the second layer of choice: granularity.

| Granularity | Number of scales | Suitable for |
|:---|:---|:---|
| per-tensor | 1 | shared across the whole graph, simplest to implement |
| per-channel | number of output channels | each row independent, most common for weight quantization |
| per-group | rows × (input_dim / group_size) | standard for 4-bit quantization, group_size usually 128 |

Below we hand-calculate the difference between the three granularities on a 4×8 weight matrix.


In [ ]:
# Build a weight matrix whose rows have very different scales
np.random.seed(7)
W = np.random.randn(4, 8).astype(np.float32)
# The second row has a noticeably larger range than the others (simulating an outlier row)
W[1] *= 8.0
print('Weight matrix W (note that row 2 has a clearly larger range):')
print(np.round(W, 2))

def mae(a, b):
    return np.mean(np.abs(a - b))

# All three granularities do symmetric INT8 quantization
def quantize_sym(W, axis=None):
    if axis is None:
        scale = np.max(np.abs(W)) / 127
    else:
        # take max along the specified axis, keep shape for broadcasting
        scale = np.max(np.abs(W), axis=axis, keepdims=True) / 127
    W_q = np.round(W / scale).clip(-127, 127)
    return W_q * scale

W_per_tensor = quantize_sym(W, axis=None)
W_per_channel = quantize_sym(W, axis=1)  # one scale per row

# per-group: split each row into chunks of group_size=4, one scale per chunk
group_size = 4
W_per_group = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(0, W.shape[1], group_size):
        chunk = W[i, j:j+group_size]
        s = np.max(np.abs(chunk)) / 127
        W_per_group[i, j:j+group_size] = np.round(chunk / s).clip(-127, 127) * s

print(f'\n{"Granularity":<15} {"MAE":>10}')
print('-' * 28)
print(f'{"per-tensor":<15} {mae(W, W_per_tensor):>10.4f}')
print(f'{"per-channel":<15} {mae(W, W_per_channel):>10.4f}')
print(f'{"per-group(4)":<15} {mae(W, W_per_group):>10.4f}')
print(f'\nKey observation: per-channel greatly reduces the error compared with per-tensor, and per-group drops it one more level.')


## 4. The Difficulty of Activation Quantization: Outlier Channels

Everything above was about weight quantization. But during inference we have $y = Wx$; if we only quantize $W$ to INT4 and keep $x$ in FP16, the matrix multiplication is still FP16 and we save little computation. To truly accelerate, $x$ must be quantized too.

Activation quantization is much harder than weight quantization. The reason is a very specific phenomenon in LLM activations: a small number of channels have values tens of times larger than the others. These are called outlier channels. If we use one scale for the whole layer, the outliers force the scale to be huge, and all the normal channels get crushed into a few values.

Below we simulate a typical LLM activation: 99% of the channels have values in $[-3, 3]$, and 1% reach $\pm 50$.


In [ ]:
np.random.seed(11)

# 256 channels, simulating the LLM activation distribution
n_channels = 256
x_act = np.random.randn(n_channels).astype(np.float32) * 1.0

# Pick 3 channels as outliers
outlier_idx = [42, 113, 201]
x_act[outlier_idx] *= 50.0

print(f'Overall channel range: [{x_act.min():.2f}, {x_act.max():.2f}] (including outliers)')
mask = np.ones(n_channels, dtype=bool)
mask[outlier_idx] = False
print(f'  Normal channels: [{x_act[mask].min():.2f}, {x_act[mask].max():.2f}]')
print(f'  Outlier channel values: {x_act[outlier_idx]}')

# Per-tensor symmetric quantization
scale = np.max(np.abs(x_act)) / 127
x_q = np.round(x_act / scale).clip(-127, 127).astype(np.int8)
x_deq = x_q.astype(np.float32) * scale

# Look at the error of normal channels and outliers separately
err_normal = np.mean(np.abs(x_act[mask] - x_deq[mask]))
err_outlier = np.mean(np.abs(x_act[outlier_idx] - x_deq[outlier_idx]))
print(f'\nAfter per-tensor quantization:')
print(f'  Normal channel average error: {err_normal:.4f} (the values themselves are only on the order of ±3, so this error is very large)')
print(f'  Outlier channel error: {err_outlier:.4f}')
print(f'\nKey observation: the outliers stretch the scale, and normal channels are left with only 2-3 effective bits.')


## 5. Intuition Behind GPTQ: Compensating Error with Second-Order Information

RTN (direct rounding) accumulates quantization error. The core idea of GPTQ is: after quantizing one weight, "compensate" the error it introduces onto the weights that have not yet been quantized, so as to minimize the total error.

The derivation uses a classical result: to minimize $\|Wx - \hat{W}x\|^2$ where $x$ follows some distribution, the optimal compensation can be computed from the covariance information of $W$ (specifically the Hessian matrix $H = X X^T$). This is where "second-order information" comes from — using the statistical properties of the input data to guide the direction in which the weights are adjusted.

The full GPTQ algorithm quantizes weights column by column and uses a Cholesky decomposition for numerical stability, which we will not expand on here. Below we only demonstrate a minimal version of the compensation idea: after quantizing one weight, spread the error proportionally over the remaining weights.


In [ ]:
np.random.seed(3)

# A 1D weight sequence
w = np.array([0.7, -1.2, 2.1, 0.4, -0.9], dtype=np.float32)
scale = np.max(np.abs(w)) / 7  # 4-bit quantization (7 = 2^3 - 1)

# Method 1: naive RTN (direct rounding)
w_rtn = np.round(w / scale).clip(-7, 7) * scale

# Method 2: sequential quantization + error compensation (minimal GPTQ)
# Pretend H = identity matrix (in practice H comes from calibration data)
w_gptq = w.copy()
w_quantized = np.zeros_like(w)
for i in range(len(w)):
    # Quantize the i-th weight
    q = np.round(w_gptq[i] / scale).clip(-7, 7)
    w_quantized[i] = q * scale
    # Error
    err = w_gptq[i] - q * scale
    # Spread the error over the remaining weights (simplified with even spreading)
    if i + 1 < len(w):
        w_gptq[i+1:] += err / (len(w) - i - 1)

print(f'{"Original":>8} {"RTN":>8} {"GPTQ minimal":>14}')
print('-' * 34)
for i in range(len(w)):
    print(f'{w[i]:>8.2f} {w_rtn[i]:>8.2f} {w_quantized[i]:>14.2f}')

err_rtn = np.sum((w - w_rtn) ** 2)
err_gptq = np.sum((w - w_quantized) ** 2)
print(f'\nTotal squared error RTN:         {err_rtn:.4f}')
print(f'Total squared error GPTQ minimal: {err_gptq:.4f}')
print(f'\nKey observation: after compensation the total error is usually lower (the minimal version uses even spreading; real GPTQ uses Hessian weighting).')


## 6. Intuition Behind AWQ: Protecting Important Channels

AWQ (Activation-aware Weight Quantization) approaches the outlier problem from another angle. Observation: the channels with large activation values (outlier channels) also contribute most to $y = Wx$ — even a small error in the weights of these channels has a large impact on the output.

AWQ's approach is: identify these "important channels" (using calibration data to see which channels have large activations), multiply their weights by a scaling factor $s > 1$ (magnifying them, equivalent to giving them finer quantization granularity), and on the activation side multiply by $1/s$ to cancel out, keeping $Wx$ numerically unchanged.

The formula: $y = Wx = (W \cdot s)(x / s)$, where $s$ takes larger values on the important channels. This way the relative quantization error of the important channel weights is reduced, and overall accuracy improves.

Below we use a simplified toy experiment to demonstrate this mechanism.


In [ ]:
# The activation from the previous section: 3 outlier channels
scale_int4 = np.max(np.abs(x_act)) / 7  # 4-bit symmetric quantization

# Method 1: naive RTN
x_q_rtn = np.round(x_act / scale_int4).clip(-7, 7) * scale_int4
err_rtn = np.mean(np.abs(x_act - x_q_rtn))

# Method 2: AWQ idea — multiply the outlier channels by an s, then divide back after quantization
s = np.ones(n_channels, dtype=np.float32)
s[outlier_idx] = 4.0  # magnify important channels by 4x

# Equivalent to quantizing (x * s), but since we are quantizing x itself (for the demo),
# we directly look at the quantization error after magnifying and dividing back
x_scaled = x_act * s
scale_scaled = np.max(np.abs(x_scaled)) / 7
x_q_scaled = np.round(x_scaled / scale_scaled).clip(-7, 7) * scale_scaled
x_q_awq = x_q_scaled / s

err_awq = np.mean(np.abs(x_act - x_q_awq))
err_normal_awq = np.mean(np.abs(x_act[mask] - x_q_awq[mask]))
err_normal_rtn = np.mean(np.abs(x_act[mask] - x_q_rtn[mask]))

print(f'Overall MAE:')
print(f'  RTN:  {err_rtn:.4f}')
print(f'  AWQ:  {err_awq:.4f}')
print(f'\nNormal channel MAE:')
print(f'  RTN:  {err_normal_rtn:.4f}')
print(f'  AWQ:  {err_normal_awq:.4f}')
print(f'\nKey observation: by scaling outliers separately, AWQ greatly improves the quantization precision of normal channels.')
print(f'Real AWQ also searches for the optimal s value and only scales the top 1% important channels, further reducing the error.')


## 7. Comparison of Mainstream Quantization Schemes

Summarizing the algorithms discussed above into one table makes the design tradeoffs easy to compare:

| Scheme | Quantization target | Core idea | Calibration need |
|:---|:---|:---|:---|
| RTN | weights / activations | direct rounding | none |
| GPTQ | weights | compensate quantization error with Hessian information | needs a small amount of calibration data |
| AWQ | weights | identify important channels and protect them by scaling | needs a small amount of calibration data |
| SmoothQuant | weights + activations | migrate the difficulty from activations to the weight side | needs a small amount of calibration data |
| QAT (quantization-aware training) | weights + activations | simulate quantization error during training so the model adapts | requires training |

The practical engineering choice: 4-bit weight quantization + FP16 activations is currently the most mainstream configuration for LLM inference (such as llama.cpp's Q4_K_M and vLLM's AWQ mode). Weight-only quantization saves memory and weight read bandwidth, which already addresses most of the bottleneck in the decode phase; activation quantization has much higher engineering complexity and is usually left to dedicated research scenarios.


## Summary

- [ ] Be able to write the formulas for symmetric and asymmetric quantization, and explain the role of the zero point
- [ ] Understand the tradeoff between precision and parameter overhead across per-tensor, per-channel, and per-group granularities
- [ ] Be able to explain why activation quantization is harder than weight quantization, and how outlier channels break the scale
- [ ] Be able to state in one sentence the core idea of GPTQ (compensate error with the Hessian) and AWQ (protect important channels by scaling)
- [ ] Know that the current mainstream 4-bit LLM inference configuration is weight-only quantization (such as Q4_K_M, AWQ)


## Exercises

The three exercises below help you put the core algorithms of this section into practice. You can ask an AI to help break down steps or check ideas, but it is not recommended to let an AI write the complete answer directly — the details of quantization only stick after you have written them once yourself.

**Exercise 1: Hand-calculate asymmetric quantization**

Given the vector $x = [0.5, 1.0, 1.5, 2.0, 2.5]$, perform INT4 asymmetric quantization (uint4, range $[0, 15]$). By hand, compute $x_{\min}$, $x_{\max}$, scale, and zero_point, write out the quantized integer and the dequantized result for each element, and compute the MAE.

Hint: first compute $s = (x_{\max} - x_{\min}) / 15$, then use $q = \text{round}((x - x_{\min}) / s)$.

**Exercise 2: Implement per-channel symmetric quantization**

Implement a function `quantize_per_channel(W, n_bits=4)` whose input is a weight matrix of shape `(out_features, in_features)`, and which independently performs symmetric quantization on each row. Return the quantized-then-dequantized floating-point matrix.


In [ ]:
# Reference answer skeleton for Exercise 2: replace the None below with your implementation

def quantize_per_channel(W, n_bits=4):
    # Your code: perform symmetric quantization on each row
    # Hint: use np.max(np.abs(W), axis=1, keepdims=True) to get the per-row max
    n_levels = 2 ** (n_bits - 1) - 1
    scale = None  # TODO: one scale per row
    W_q = None    # TODO: round + clip
    return W_q * scale

# Verification
np.random.seed(42)
W_test = np.random.randn(8, 16).astype(np.float32)
W_test[3] *= 5  # one outlier row

W_restored = quantize_per_channel(W_test, n_bits=4)
assert W_restored.shape == W_test.shape, 'Shape mismatch'
mae_test = np.mean(np.abs(W_test - W_restored))
assert mae_test < 0.2, f'MAE too large: {mae_test:.3f} (per-channel 4-bit can usually be pushed below 0.1)'
print(f'Exercise 2 passed: per-channel 4-bit MAE = {mae_test:.4f}')
print('  You can already write half the logic of a production-grade 4-bit quantizer.')


**Exercise 3: Compare the precision loss of 4-bit vs 8-bit**

Use `quantize_per_channel` to quantize the same $W$ at both 4-bit and 8-bit, and compare the MAE. Then think about this question: if you shrink the granularity from per-channel down to per-group (group_size=16), can the 4-bit MAE get close to the level of 8-bit per-channel? Write a short piece of code to verify.

Hint: for 8-bit, `n_levels = 127`; for 4-bit, `n_levels = 7`. For the group_size implementation, refer to cell 6 in the main text.


## References

- Frantar et al., [GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers](https://arxiv.org/abs/2210.17323), 2022
- Lin et al., [AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration](https://arxiv.org/abs/2306.00978), 2023
- Xiao et al., [SmoothQuant: Accurate and Efficient Post-Training Quantization for Large Language Models](https://arxiv.org/abs/2211.10438), 2022
- Dettmers et al., [LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale](https://arxiv.org/abs/2208.07339), 2022 — the first paper to systematically discuss LLM activation outliers
- Jacob et al., [Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference](https://arxiv.org/abs/1712.05877), 2017 — the classic source of the asymmetric quantization formula
